# ArSL Word — Live Webcam Testing

This notebook lets you **test your trained Arabic sign language word model in real-time** using your webcam.

### How it works:

1. **Continuous capture** — MediaPipe extracts hand landmarks every frame (supports 1 or 2 hands)
2. **Sliding window** — buffers the last 30 frames into a sequence
3. **Prediction** — feeds the sequence to the BiLSTM model every 0.5s
4. **Sentence building** — confirmed words are appended to a sentence

### Two-Hand Support:

- **Auto-detects** the model's expected input shape (63 or 126 features)
- If the model expects **63 features** (1 hand) — uses the dominant hand only
- If the model expects **126 features** (2 hands) — captures both hands and concatenates landmarks
- Many Arabic sign language word signs require two hands for proper recognition

### Controls:

| Key         | Action                  |
| ----------- | ----------------------- |
| `q`         | Quit                    |
| `r`         | Reset sentence          |
| `SPACE`     | Add space between words |
| `BACKSPACE` | Delete last word        |

### Requirements:

- Trained model: `arsl_word_lstm_model_best.h5`
- Class mapping: `arsl_word_classes.csv`
- Webcam connected


In [28]:
# ===============================
# CELL 1: IMPORTS & SETUP
# ===============================

import cv2
import json
import time
import numpy as np
import pandas as pd
import mediapipe as mp
import tensorflow as tf
from pathlib import Path
from collections import deque

# Arabic text rendering (fixes ??? in window)
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display

print(f'TensorFlow: {tf.__version__}')
print(f'OpenCV: {cv2.__version__}')
print(f'MediaPipe: {mp.__version__}')

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU detected: {gpus[0].name}')
else:
    print('No GPU — running on CPU')

# --- Arabic-safe text drawing helper ---
_ARABIC_FONT_PATH = 'C:/Windows/Fonts/arial.ttf'
_arabic_font_cache = {}

def draw_arabic_text(frame, text, pos, font_size=26, color=(255, 255, 255)):
    """Draw Arabic (or mixed) text correctly on an OpenCV frame using PIL."""
    try:
        reshaped = arabic_reshaper.reshape(text)
        bidi_text = get_display(reshaped)
    except Exception:
        bidi_text = text

    if font_size not in _arabic_font_cache:
        try:
            _arabic_font_cache[font_size] = ImageFont.truetype(_ARABIC_FONT_PATH, font_size)
        except Exception:
            _arabic_font_cache[font_size] = ImageFont.load_default()
    font = _arabic_font_cache[font_size]

    pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil_img)
    # PIL color is RGB; OpenCV color is BGR
    rgb_color = (color[2], color[1], color[0])
    draw.text(pos, bidi_text, font=font, fill=rgb_color)
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)


TensorFlow: 2.10.0
OpenCV: 4.11.0
MediaPipe: 0.10.9
GPU detected: /physical_device:GPU:0


In [29]:
# ===============================
# CELL 2: CONFIGURATION
# ===============================
import tkinter as tk
from tkinter import filedialog, ttk
from pathlib import Path
import json

CONFIG_FILE = "live_test_config_cache.json"

def get_live_test_config():
    """Launches a popup UI to manually configure the live testing session."""
    config = {}
    
    # Load cached config if exists
    cache = {}
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE, "r") as f:
                cache = json.load(f)
        except Exception as e:
            print(f"Warning: Could not load config cache: {e}")

    root = tk.Tk()
    root.title("Live Test Configuration")
    root.geometry("600x250")
    root.eval('tk::PlaceWindow . center')

    # Variables
    var_model = tk.StringVar(value=cache.get("MODEL_PATH", ""))
    var_classes = tk.StringVar(value=cache.get("CLASSES_CSV", ""))

    def browse_h5(var):
        file = filedialog.askopenfilename(filetypes=[("H5 Model", "*.h5")])
        if file: var.set(file)

    def browse_csv(var):
        file = filedialog.askopenfilename(filetypes=[("CSV Files", "*.csv")])
        if file: var.set(file)

    ttk.Label(root, text="Model File (.h5):").grid(row=0, column=0, sticky="w", padx=10, pady=15)
    ttk.Entry(root, textvariable=var_model, width=40).grid(row=0, column=1, padx=10)
    ttk.Button(root, text="Browse", command=lambda: browse_h5(var_model)).grid(row=0, column=2)

    ttk.Label(root, text="Classes CSV:").grid(row=1, column=0, sticky="w", padx=10, pady=15)
    ttk.Entry(root, textvariable=var_classes, width=40).grid(row=1, column=1, padx=10)
    ttk.Button(root, text="Browse", command=lambda: browse_csv(var_classes)).grid(row=1, column=2)

    def start():
        config['MODEL_PATH'] = Path(var_model.get()) if var_model.get() else None
        config['CLASSES_CSV'] = Path(var_classes.get()) if var_classes.get() else None
        
        # Save to cache
        cache_data = {
            'MODEL_PATH': var_model.get(),
            'CLASSES_CSV': var_classes.get()
        }
        try:
            with open(CONFIG_FILE, "w") as f:
                json.dump(cache_data, f, indent=4)
        except Exception as e:
            print(f"Warning: Could not save config cache: {e}")
            
        root.destroy()

    ttk.Button(root, text="✅ SAVE CONFIG & CONTINUE", command=start).grid(row=2, column=0, columnspan=3, pady=30)

    root.mainloop()
    return config

print("Launching Configuration Panel...")
C = get_live_test_config()

MODEL_PATH = C['MODEL_PATH']
CLASSES_CSV = C['CLASSES_CSV']

if not MODEL_PATH or not CLASSES_CSV:
    raise ValueError("❌ You must select all files in the popup!")


# Sequence parameters (must match training)
SEQUENCE_LENGTH = 30    # frames per sequence

# Hand detection mode: auto-detected from model input shape
# - 63 features = 1 hand (21 landmarks x 3)
# - 126 or 258 features = 2 hands
# Set to None for auto-detection, or override manually:
NUM_FEATURES = None  # will be set after model loads

# Live inference settings
CONFIDENCE_THRESHOLD = 0.35     # minimum confidence to accept a prediction
PREDICTION_INTERVAL = 0.5       # seconds between predictions
STABILITY_WINDOW = 3            # consecutive same predictions needed to confirm
COOLDOWN_TIME = 2.0             # seconds after confirming a word before next

# Camera: capture small for speed, display large for comfort
CAMERA_INDEX = 0
CAMERA_WIDTH  = 640    # capture resolution (smaller = faster FPS)
CAMERA_HEIGHT = 480
DISPLAY_WIDTH  = 1280  # display resolution (upscaled after capture)
DISPLAY_HEIGHT = 720

print(f'Model  : {MODEL_PATH}')
print(f'Classes: {CLASSES_CSV}')
print(f'Sequence: {SEQUENCE_LENGTH} frames')
print(f'Confidence threshold: {CONFIDENCE_THRESHOLD}')
print(f'Stability window: {STABILITY_WINDOW} predictions')


Launching Configuration Panel...
Model  : M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\arsl_word_lstm_model_final_v2.h5
Classes: M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\arsl_word_classes.csv
Sequence: 30 frames
Confidence threshold: 0.35
Stability window: 3 predictions


In [30]:
# ===============================
# CELL 3: LOAD MODEL, VOCABULARY & SCALER
# ===============================
import numpy as np

# --- Custom layer needed for model loading ---
class TemporalAttention(tf.keras.layers.Layer):
    """Temporal attention layer (must match training definition)."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight', shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='att_bias', shape=(input_shape[1], 1),
                                 initializer='zeros', trainable=True)

    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = tf.reduce_sum(x * a, axis=1)
        return output

# Load model
print('Loading model...')
model = tf.keras.models.load_model(
    str(MODEL_PATH),
    custom_objects={'TemporalAttention': TemporalAttention}
)
print(f'Model loaded: {model.name} — {model.count_params():,} parameters')

# Auto-detect feature count from model input shape
model_input_shape = model.input_shape  # (None, SEQUENCE_LENGTH, NUM_FEATURES)
NUM_FEATURES = model_input_shape[-1]
NUM_HANDS = 2 if (NUM_FEATURES == 126 or NUM_FEATURES == 258) else 1
LANDMARKS_PER_HAND = 21 * 3  # 63

print(f'Model expects {NUM_FEATURES} features -> {NUM_HANDS} hand(s) mode')

# Load Scaler Stats
SCALER_PATH = Path(MODEL_PATH).parent / "arsl_scaler_stats.npz"
if SCALER_PATH.exists():
    z = np.load(SCALER_PATH)
    scaler_mean = z['mean']
    scaler_scale = z['scale']
    print(f"✅ Loaded Scaler: {SCALER_PATH.name}")
else:
    print(f"⚠️ WARNING: No scaler found at {SCALER_PATH} - Prediction might be inaccurate!")
    scaler_mean = 0.0
    scaler_scale = 1.0

# Load class mapping directly from the CSV generated by training
class_df = pd.read_csv(CLASSES_CSV)

# Build model_index -> word name mapping (Since no shared vocab is used, both EN and AR will show the label_name)
index_to_english = {}
index_to_arabic = {}
for _, row in class_df.iterrows():
    idx = int(row['model_class_index'])
    label = str(row['label_name'])
    
    # We use label for both since there is no separate translation file
    index_to_english[idx] = label
    index_to_arabic[idx] = label

num_classes = len(index_to_english)
print(f'{num_classes} word classes loaded')
print(f'\nSample words:')
for i in list(index_to_english.keys())[:10]:
    print(f'   {i}: {index_to_english[i]}')


Loading model...
Model loaded: sequential — 512,574 parameters
Model expects 258 features -> 2 hand(s) mode
✅ Loaded Scaler: arsl_scaler_stats.npz
190 word classes loaded

Sample words:
   0: 0
   1: 1
   2: 10
   3: 100
   4: 1000
   5: 1000000
   6: 10000000
   7: 2
   8: 20
   9: 200


In [31]:
# ===============================
# CELL 4: MEDIAPIPE DETECTOR
# ===============================
# Supports 63 features (1 hand), 126 features (2 hands), and 258 features (Holistic)

mp_hands = mp.solutions.hands
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Initialize appropriate detector based on features
if NUM_FEATURES == 258:
    detector = mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=0,  # 0 = fastest, minimal accuracy loss for pose
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    print("MediaPipe Holistic detector ready (258 features mode, complexity=0)")
else:
    detector = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=NUM_HANDS,
        min_detection_confidence=0.6,
        min_tracking_confidence=0.6
    )
    print(f'MediaPipe hand detector ready ({NUM_HANDS} hand(s) mode)')

def extract_landmarks(frame):
    """Extract landmarks dynamically based on NUM_FEATURES."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    draw_landmarks = []
    
    if NUM_FEATURES == 258:
        results = detector.process(rgb)
        
        # 1. Pose: 33 x 4 = 132 features
        if results.pose_landmarks:
            pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark], dtype=np.float32).flatten()
            draw_landmarks.append(('pose', results.pose_landmarks))
        else:
            pose = np.zeros(132, dtype=np.float32)

        # 2. Left hand: 21 x 3 = 63 features
        if results.left_hand_landmarks:
            lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark], dtype=np.float32).flatten()
            draw_landmarks.append(('hand', results.left_hand_landmarks))
        else:
            lh = np.zeros(63, dtype=np.float32)

        # 3. Right hand: 21 x 3 = 63 features
        if results.right_hand_landmarks:
            rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark], dtype=np.float32).flatten()
            draw_landmarks.append(('hand', results.right_hand_landmarks))
        else:
            rh = np.zeros(63, dtype=np.float32)

        combined = np.concatenate([pose, lh, rh])
        return combined, draw_landmarks

    elif NUM_HANDS == 1: # 63 features
        results = detector.process(rgb)
        if results.multi_hand_landmarks:
            lm = results.multi_hand_landmarks[0]
            vec = np.array([[p.x, p.y, p.z] for p in lm.landmark], dtype=np.float32).flatten()
            return vec, [('hand', lm)]
        return np.zeros(NUM_FEATURES, dtype=np.float32), []

    else: # 126 features
        results = detector.process(rgb)
        left_vec = np.zeros(LANDMARKS_PER_HAND, dtype=np.float32)
        right_vec = np.zeros(LANDMARKS_PER_HAND, dtype=np.float32)

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_lm, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                draw_landmarks.append(('hand', hand_lm))
                label = handedness.classification[0].label
                vec = np.array([[p.x, p.y, p.z] for p in hand_lm.landmark], dtype=np.float32).flatten()

                if label == 'Left':
                    left_vec = vec
                else:
                    right_vec = vec

        combined = np.concatenate([left_vec, right_vec])
        return combined, draw_landmarks


MediaPipe Holistic detector ready (258 features mode, complexity=0)


In [32]:
# ===============================
# CELL 5: LIVE WEBCAM TESTING
# ===============================

FONT = cv2.FONT_HERSHEY_SIMPLEX

def run_live_test():
    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    cap.set(cv2.CAP_PROP_FPS, 30)

    if not cap.isOpened():
        print('Cannot open camera!'); return

    print(f'Camera opened [{NUM_HANDS}H/{NUM_FEATURES}F].')
    print('(Q) Quit  (R/C) Clear  (SPACE) Space  (D/BACKSPACE) Delete')

    frame_buffer       = deque(maxlen=SEQUENCE_LENGTH)
    prediction_history = deque(maxlen=STABILITY_WINDOW)
    sentence_words_en  = []
    sentence_words_ar  = []
    current_word_en    = ''
    current_word_ar    = ''
    current_conf       = 0.0
    last_pred_time     = 0.0
    last_conf_time     = 0.0
    last_erase_time    = 0.0
    hand_detected      = False
    hands_count        = 0
    fps_history        = deque(maxlen=30)
    top3               = []

    # ── Colors (BGR) ──
    GREEN  = (0, 200, 0)
    RED    = (0, 0, 200)
    WHITE  = (255, 255, 255)
    BLACK  = (0, 0, 0)
    YELLOW = (0, 210, 210)
    ORANGE = (0, 140, 255)
    GRAY   = (70, 70, 70)
    LGRAY  = (160, 160, 160)

    # ── Layout constants (absolute, for 1280x720 display) ──
    TOP_H   = 115   # top bar height
    BOT_H   = 82    # bottom bar height
    # Zone x boundaries inside top bar:
    #   LEFT   0  ..  470  : Word + Confidence %
    #   CENTER 485 ..  700  : Progress bars
    #   RIGHT  715 .. 1280  : Top-3 predictions
    L_END  = 470
    C_X    = 485
    C_END  = 700
    R_X    = 715

    while True:
        t0 = time.time()
        ret, frame = cap.read()
        if not ret: break

        frame = cv2.flip(frame, 1)
        # Upscale BEFORE drawing so UI is crisp at display resolution
        frame = cv2.resize(frame, (DISPLAY_WIDTH, DISPLAY_HEIGHT))
        h, w = frame.shape[:2]   # 720, 1280
        now  = time.time()

        # ── Landmark extraction ──────────────────────────────────────────────
        landmarks, draw_items = extract_landmarks(frame)
        hands_count  = sum(1 for t, _ in draw_items if t == 'hand')
        hand_detected = hands_count > 0
        frame_buffer.append(landmarks)

        # ── Erase gesture ────────────────────────────────────────────────────
        if NUM_FEATURES == 258 and (now - last_erase_time) > 2.0:
            ls_x, ls_y = landmarks[44], landmarks[45]
            rs_x, rs_y = landmarks[48], landmarks[49]
            lw_x, lw_y = landmarks[60], landmarks[61]
            rw_x, rw_y = landmarks[64], landmarks[65]
            if ls_x and rs_x and lw_x and rw_x:
                d1 = ((lw_x-rs_x)**2 + (lw_y-rs_y)**2)**0.5
                d2 = ((rw_x-ls_x)**2 + (rw_y-ls_y)**2)**0.5
                if d1 < 0.2 and d2 < 0.2 and sentence_words_en:
                    sentence_words_en.pop()
                    if sentence_words_ar: sentence_words_ar.pop()
                    cv2.putText(frame, 'WORD ERASED!', (w//2-180, h//2),
                                FONT, 2.0, RED, 4)
                    print('Erase gesture: deleted last word.')
                    last_erase_time = now
                    frame_buffer.clear()

        # ── Draw landmarks ───────────────────────────────────────────────────
        for item_type, lm in draw_items:
            if item_type == 'hand':
                mp_drawing.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style())
            elif item_type == 'pose':
                mp_drawing.draw_landmarks(frame, lm, mp_holistic.POSE_CONNECTIONS,
                    mp_drawing_styles.get_default_pose_landmarks_style())

        # ── Prediction ───────────────────────────────────────────────────────
        if len(frame_buffer) == SEQUENCE_LENGTH and (now - last_pred_time) >= PREDICTION_INTERVAL:
            last_pred_time = now
            seq = np.array(list(frame_buffer), dtype=np.float32)
            seq = (seq - scaler_mean) / scaler_scale
            seq = np.expand_dims(seq, 0)
            nz  = np.sum(np.any(seq[0] != 0, axis=1))
            if nz >= SEQUENCE_LENGTH * 0.3:
                proba    = model(seq, training=False).numpy()[0]
                pred_idx = int(np.argmax(proba))
                pred_conf = float(proba[pred_idx])
                pred_en  = index_to_english.get(pred_idx, '?')
                pred_ar  = index_to_arabic.get(pred_idx, '?')
                top3_idx = np.argsort(proba)[-3:][::-1]
                top3 = [(index_to_english.get(i,'?'), index_to_arabic.get(i,'?'), float(proba[i]))
                        for i in top3_idx]
                if pred_conf >= CONFIDENCE_THRESHOLD:
                    current_word_en = pred_en
                    current_word_ar = pred_ar
                    current_conf    = pred_conf
                    prediction_history.append(pred_en)
                    if (len(prediction_history) == STABILITY_WINDOW and
                        len(set(prediction_history)) == 1 and
                        (now - last_conf_time) >= COOLDOWN_TIME):
                        sentence_words_en.append(current_word_en)
                        sentence_words_ar.append(current_word_ar)
                        last_conf_time = now
                        prediction_history.clear()
                        print(f'Confirmed: "{current_word_ar}" ({current_conf:.1%})')
                else:
                    current_word_en = current_word_ar = ''
                    current_conf = 0.0
            else:
                current_word_en = current_word_ar = ''
                current_conf = 0.0

        # ════════════════════════════════════════════════════════════════
        #  TOP HUD  (three non-overlapping zones)
        # ════════════════════════════════════════════════════════════════
        cv2.rectangle(frame, (0,0), (w, TOP_H), BLACK, -1)
        cv2.rectangle(frame, (0,0), (w, TOP_H), WHITE, 2)
        cv2.line(frame, (L_END+7, 10), (L_END+7, TOP_H-10), GRAY, 1)
        cv2.line(frame, (C_END+7, 10), (C_END+7, TOP_H-10), GRAY, 1)

        if current_word_en:
            clr = GREEN if current_conf >= 0.6 else YELLOW if current_conf >= 0.4 else ORANGE

            # ── LEFT: 'Word:' label (cv2) + Arabic word (PIL) on same line ──
            cv2.putText(frame, 'Word:', (14, 43), FONT, 1.05, clr, 2)
            word_label = current_word_ar if current_word_ar else current_word_en
            frame = draw_arabic_text(frame, word_label, (130, 6), font_size=36, color=clr)

            # 'Confidence:' label (cv2) + value (cv2) on second line
            cv2.putText(frame, f'Confidence: {current_conf:.1%}', (14, 85), FONT, 0.75, clr, 2)

            # ── CENTER: confidence bar + stability bar ───────────────────────
            BAR_W = C_END - C_X - 8

            # Confidence bar
            cv2.putText(frame, 'Conf', (C_X, 18), FONT, 0.46, LGRAY, 1)
            BY1 = 22; BH1 = 24
            cv2.rectangle(frame, (C_X, BY1), (C_X+BAR_W, BY1+BH1), GRAY, -1)
            cv2.rectangle(frame, (C_X, BY1), (C_X+int(BAR_W*current_conf), BY1+BH1), clr, -1)
            cv2.rectangle(frame, (C_X, BY1), (C_X+BAR_W, BY1+BH1), WHITE, 1)

            # Stability bar
            stable = sum(1 for p in prediction_history if p == current_word_en)
            BY2 = BY1 + BH1 + 22
            cv2.putText(frame, f'Stability  {stable}/{STABILITY_WINDOW}',
                        (C_X, BY2 - 4), FONT, 0.46, LGRAY, 1)
            BH2 = 16
            cv2.rectangle(frame, (C_X, BY2), (C_X+BAR_W, BY2+BH2), GRAY, -1)
            cv2.rectangle(frame, (C_X, BY2),
                          (C_X+int(BAR_W*stable/max(STABILITY_WINDOW,1)), BY2+BH2), WHITE, -1)
            cv2.rectangle(frame, (C_X, BY2), (C_X+BAR_W, BY2+BH2), WHITE, 1)

            # ── RIGHT: Top-3  (rank=cv2 | Arabic=PIL | pct=cv2) ─────────────
            # Separate rendering avoids BiDi flipping rank numbers and percentages
            cv2.putText(frame, 'Top 3:', (R_X, 16), FONT, 0.50, LGRAY, 1)
            PCT_X = w - 8   # right-align percentages here
            for rank, (tw_en, tw_ar, tc) in enumerate(top3):
                label    = tw_ar if tw_ar else tw_en
                y_pil    = 20 + rank * 32          # PIL: top-left of text
                y_cv     = y_pil + 22              # cv2: baseline
                rank_clr = WHITE if rank == 0 else LGRAY
                pct_str  = f'{tc:.1%}'

                # Rank number at left of right zone
                cv2.putText(frame, f'{rank+1}.', (R_X, y_cv), FONT, 0.55, rank_clr, 1)

                # Percentage at fixed right column (right-aligned)
                (ptw, _), _ = cv2.getTextSize(pct_str, FONT, 0.52, 1)
                cv2.putText(frame, pct_str, (PCT_X - ptw, y_cv), FONT, 0.52, LGRAY, 1)

                # Arabic word in the space between rank and percentage
                arabic_x   = R_X + 32
                arabic_end = PCT_X - ptw - 12
                frame = draw_arabic_text(frame, label, (arabic_x, y_pil),
                                         font_size=20, color=rank_clr)
        else:
            status = 'Show a sign...' if hand_detected else 'No hand detected'
            cv2.putText(frame, status, (15, 62), FONT, 1.1, LGRAY, 2)

        # ════════════════════════════════════════════════════════════════
        #  BOTTOM SENTENCE BAR
        # ════════════════════════════════════════════════════════════════
        cv2.rectangle(frame, (0, h-BOT_H), (w, h), BLACK, -1)
        cv2.rectangle(frame, (0, h-BOT_H), (w, h), WHITE, 2)

        sent_ar = ' '.join(sentence_words_ar) if sentence_words_ar else '(sentence will appear here)'
        sent_en = ' '.join(sentence_words_en) if sentence_words_en else ''

        # 'AR:' label (cv2) stays LTR; Arabic sentence (PIL) appears after it
        cv2.putText(frame, 'AR:', (12, h-BOT_H+34), FONT, 0.78, (80,220,80), 2)
        frame = draw_arabic_text(frame, sent_ar, (68, h-BOT_H+6), font_size=26, color=(80,220,80))

        cv2.putText(frame, 'EN:', (12, h-BOT_H+68), FONT, 0.70, WHITE, 1)
        frame = draw_arabic_text(frame, sent_en, (68, h-BOT_H+44), font_size=22, color=WHITE)

        # ════════════════════════════════════════════════════════════════
        #  OVERLAYS on video area
        # ════════════════════════════════════════════════════════════════
        # Buffer (bottom-left above bar)
        buf_clr = GREEN if len(frame_buffer) >= SEQUENCE_LENGTH else YELLOW
        cv2.putText(frame, f'Buffer: {len(frame_buffer)}/{SEQUENCE_LENGTH}',
                    (12, h-BOT_H-12), FONT, 0.55, buf_clr, 1)

        # Hand status (bottom-right above bar)
        if NUM_HANDS == 2:
            h_clr  = GREEN if hands_count==2 else YELLOW if hands_count==1 else RED
            h_txt  = f'HANDS: {hands_count}/2' if hands_count else 'NO HANDS'
        else:
            h_clr  = GREEN if hand_detected else RED
            h_txt  = 'HAND OK' if hand_detected else 'NO HAND'
        (htw, _), _ = cv2.getTextSize(h_txt, FONT, 0.55, 1)
        cv2.putText(frame, h_txt, (w-htw-24, h-BOT_H-12), FONT, 0.55, h_clr, 1)
        cv2.circle(frame, (w-12, h-BOT_H-18), 7, h_clr, -1)

        # FPS + mode (top-right below top bar)
        fps = 1.0 / max(time.time()-t0, 1e-6)
        fps_history.append(fps)
        avg_fps = sum(fps_history)/len(fps_history)
        cv2.putText(frame, f'FPS: {avg_fps:.0f}', (w-105, TOP_H+22), FONT, 0.60, WHITE, 1)
        cv2.putText(frame, f'ArSL | {NUM_HANDS}H/{NUM_FEATURES}F',
                    (w-195, TOP_H+42), FONT, 0.48, LGRAY, 1)

        cv2.imshow('ArSL Word Recognition - Live Test', frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key in (ord('r'), ord('c')):
            sentence_words_en.clear(); sentence_words_ar.clear()
            prediction_history.clear()
            current_word_en = current_word_ar = ''
            print('Cleared.')
        elif key == 32:
            sentence_words_en.append(' '); sentence_words_ar.append(' ')
        elif key in (8, 127) or key == ord('d') or key == ord('x'):
            if sentence_words_en:
                re = sentence_words_en.pop()
                ra = sentence_words_ar.pop() if sentence_words_ar else ''
                print(f'Removed: "{re}" / "{ra}"')

    cap.release()
    cv2.destroyAllWindows()
    fe = ' '.join(sentence_words_en)
    fa = ' '.join(sentence_words_ar)
    print(f'\nFinal (EN): {fe}')
    print(f'Final (AR): {fa}')
    return fe, fa

result = run_live_test()


Camera opened [2H/258F].
(Q) Quit  (R/C) Clear  (SPACE) Space  (D/BACKSPACE) Delete
Confirmed: "1000" (84.1%)
Confirmed: "حواس خمس" (98.4%)
Confirmed: "حواس خمس" (98.7%)
Confirmed: "ح" (64.0%)
Confirmed: "جمجة" (89.9%)
Confirmed: "ذ" (55.5%)
Confirmed: "500" (65.6%)
Confirmed: "500" (61.4%)
Confirmed: "40" (90.4%)
Confirmed: "40" (48.8%)
Confirmed: "عضلة" (69.7%)
Confirmed: "عضلة" (68.2%)
Confirmed: "عضلة" (59.7%)
Confirmed: "ألم" (35.5%)
Confirmed: "ح" (38.7%)
Confirmed: "دواء" (95.7%)
Confirmed: "عضلة" (87.2%)
Confirmed: "غ" (51.5%)
Confirmed: "أخذ إبرة" (62.1%)
Confirmed: "عضلة" (59.2%)
Confirmed: "حواس خمس" (91.1%)
Confirmed: "ئـ" (71.2%)
Erase gesture: deleted last word.
Erase gesture: deleted last word.
Erase gesture: deleted last word.
Erase gesture: deleted last word.
Erase gesture: deleted last word.
Erase gesture: deleted last word.
Erase gesture: deleted last word.
Erase gesture: deleted last word.
Erase gesture: deleted last word.
Erase gesture: deleted last word.
Erase ges

## Tips

| Issue                      | Solution                                                                                       |
| -------------------------- | ---------------------------------------------------------------------------------------------- |
| **Low FPS**                | Close other apps, reduce `CAMERA_WIDTH`/`CAMERA_HEIGHT`                                        |
| **Wrong predictions**      | Hold the sign steadily for ~2 seconds                                                          |
| **Camera not opening**     | Change `CAMERA_INDEX` to 1 or 2                                                                |
| **Too sensitive**          | Increase `STABILITY_WINDOW` to 4-5                                                             |
| **Not detecting**          | Lower `CONFIDENCE_THRESHOLD` to 0.25                                                           |
| **Too slow between words** | Decrease `COOLDOWN_TIME` to 1.0                                                                |
| **Only 1 hand shown**      | The model auto-detects hand count from its input shape. Retrain with 2 hands for full support. |

### How to perform a sign:

1. Face the camera with your hand(s) clearly visible
2. Perform the Arabic sign gesture smoothly
3. Wait for the stability bar to fill up
4. The word will be confirmed and added to the sentence (both English and Arabic)

### Two-Hand Mode Notes:

- If your model was trained with 126 features (2 hands), both hands will be tracked
- Hands are ordered consistently: Left first, Right second
- If only one hand is visible, the other hand's landmarks are zero-padded
- For best results with two-hand signs, keep both hands in the camera frame

### Arabic Display:

- The bottom bar shows both **English** and **Arabic** translations
- Top-3 predictions show English names (OpenCV has limited Arabic font support)
- Full Arabic display requires a GUI framework with Arabic font rendering
